In [1]:
import os 
from glob import glob
import pandas as pd

In [2]:
# 특정 경로의 파일의 목록을 가져오는 기능 
# os 라이브러리를 이용
os.listdir('./')

['01_review.ipynb',
 '01_review_bs.ipynb',
 '1-1.여성의류(196).json',
 '1-1.여성의류(197).json',
 '1-1.여성의류(198).json',
 '1-1.여성의류(199).json',
 '1-1.여성의류(200).json',
 '1-1.여성의류(201).json',
 '1-1.여성의류(202).json',
 '1-1.여성의류(203).json',
 '1-1.여성의류(204).json',
 '1-1.여성의류(205).json',
 '1-1.여성의류(206).json',
 '1-1.여성의류(207).json',
 '1-1.여성의류(208).json',
 '1-1.여성의류(209).json']

In [3]:
# glob 이용 
# 장점 : 파일의 경로와 파일의 이름을 하나의 리스트로 생성 
#       특정 확장자만 선택해서 리스트로 생성이 가능
json_list = glob("./*.json")

In [4]:
# json_list를 이용하여 하나의 데이터프레임으로 단순 행 결합

# 빈 데이터프레임을 생성
total_df = pd.DataFrame()

for file_path in json_list:
    # print(file_path)
    df = pd.read_json(file_path)
    # total_df, df를 단순 행결합을 하여 total_df에 대입 
    total_df = pd.concat( [total_df, df], axis=0 )
    # print(df)
    # break
total_df.reset_index(drop=True, inplace=True)

In [5]:
total_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1423 entries, 0 to 1422
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            1423 non-null   int64  
 1   RawText          1423 non-null   object 
 2   Source           1423 non-null   object 
 3   Domain           1423 non-null   object 
 4   MainCategory     1423 non-null   object 
 5   ProductName      1423 non-null   object 
 6   Syllable         1423 non-null   int64  
 7   Word             1423 non-null   int64  
 8   GeneralPolarity  1418 non-null   float64
 9   Aspects          1423 non-null   object 
dtypes: float64(1), int64(3), object(6)
memory usage: 111.3+ KB


In [6]:
pd.concat(
    [ pd.read_json(file_path) for file_path in json_list[:5] ]
).info()

<class 'pandas.core.frame.DataFrame'>
Index: 523 entries, 0 to 99
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            523 non-null    int64  
 1   RawText          523 non-null    object 
 2   Source           523 non-null    object 
 3   Domain           523 non-null    object 
 4   MainCategory     523 non-null    object 
 5   ProductName      523 non-null    object 
 6   Syllable         523 non-null    int64  
 7   Word             523 non-null    int64  
 8   GeneralPolarity  519 non-null    float64
 9   Aspects          523 non-null    object 
dtypes: float64(1), int64(3), object(6)
memory usage: 44.9+ KB


In [7]:
# Aspects 의 데이터를 하나로 합치고 새로운 데이터 프레임을 생성 
aspect_df = pd.DataFrame(sum(total_df['Aspects'], []))

In [8]:
aspect_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10962 entries, 0 to 10961
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Aspect             10962 non-null  object
 1   SentimentText      10962 non-null  object
 2   SentimentWord      10962 non-null  object
 3   SentimentPolarity  10962 non-null  object
dtypes: object(4)
memory usage: 342.7+ KB


In [9]:
# 데이터의 분균형 문제 확인 
aspect_df['SentimentPolarity'].value_counts()

SentimentPolarity
1     9664
-1    1005
0      293
Name: count, dtype: int64

In [10]:
aspect_df.isna().sum()

Aspect               0
SentimentText        0
SentimentWord        0
SentimentPolarity    0
dtype: int64

In [11]:
# 데이터셋에서 문자열의 좌우의 공백을 제거 
# 모든 컬럼이 Object 형이기 때문에 strip() 바로 사용 가능
aspect_df = aspect_df.map(lambda x : x.strip())

In [12]:
aspect_df.isin(['']).sum()

Aspect               0
SentimentText        0
SentimentWord        0
SentimentPolarity    0
dtype: int64

In [13]:
aspect_df['SentimentText'].value_counts()

SentimentText
가볍고                                           38
따뜻하고                                          25
저렴한 가격에                                       15
가격도 저렴하고                                      12
시원하고                                          11
                                              ..
비싼 밍크코트의 품질을 기대하지는 마시길 바랍니다.                   1
디자인이 너무너무 예쁩니다.                                1
색상도 고급스러워서 마음에 들구요.                            1
바느질도 꼼꼼하게 잘 되어 있어요.                            1
딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서     1
Name: count, Length: 10467, dtype: int64

In [14]:
before_cnt = len(aspect_df)

aspect_df.drop_duplicates('SentimentText', inplace=True)

after_cnt = len(aspect_df)

print(f"제거가 된 행의 개수 {before_cnt - after_cnt}")

제거가 된 행의 개수 495


In [15]:
# 1, 0, -1 의 비율을 확인 
aspect_df['SentimentPolarity'].value_counts()

SentimentPolarity
1     9183
-1     994
0      290
Name: count, dtype: int64

In [16]:
# 인덱스를 초기화 
aspect_df.reset_index(drop=True, inplace=True)

In [17]:
# 토큰화 -> 백터화 
from konlpy.tag import Komoran
from sklearn.feature_extraction.text import TfidfVectorizer

komoran = Komoran()
allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'SL']

def komoran_tokenize(text):
    tokens = []
    for word, pos in komoran.pos(text):
        if (pos in allow_pos) & (len(word) >= 2)  :
            tokens.append(word)
    return tokens

vectorizer = TfidfVectorizer(
    tokenizer= komoran_tokenize, 
    ngram_range=(1, 2), 
    min_df = 3, 
    max_df=0.8, 
    max_features=30000
)

In [18]:
# 모델 생성 
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier

In [19]:
svc = LinearSVC(random_state=42, class_weight='balanced')

multi_model = MultiOutputClassifier(svc)

pipe = Pipeline(
    [
        ('vector', vectorizer), 
        ('model', multi_model)
    ]
)


In [20]:
# 계층화 폴드 
from sklearn.model_selection import KFold

skfold = KFold(n_splits=3, shuffle= True, 
                         random_state=42)

In [21]:
aspect_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10467 entries, 0 to 10466
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Aspect             10467 non-null  object
 1   SentimentText      10467 non-null  object
 2   SentimentWord      10467 non-null  object
 3   SentimentPolarity  10467 non-null  object
dtypes: object(4)
memory usage: 327.2+ KB


In [22]:
from sklearn.preprocessing import LabelEncoder

In [23]:
le = LabelEncoder()
aspect_df['Aspect'] = le.fit_transform(aspect_df['Aspect'])
aspect_df['SentimentPolarity'] = aspect_df[
    'SentimentPolarity'].astype('int')

In [24]:
# 독립 변수 , 종속 변수 생성
X = aspect_df['SentimentText'].values
Y = aspect_df[['Aspect', 'SentimentPolarity']].values

In [25]:
print(X.shape, Y.shape)

(10467,) (10467, 2)


In [26]:
print(type(Y[0][0]), type(Y[0][1]))

<class 'numpy.int64'> <class 'numpy.int64'>


In [27]:
from sklearn.model_selection import GridSearchCV

In [28]:
params = {
    'model__estimator__C' : [1.0, 2.0]
}
grid = GridSearchCV(
    estimator=pipe, 
    param_grid= params, 
    cv = skfold, 
    scoring="accuracy"
)

In [29]:
grid.fit(X, Y)

c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py:953: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 942, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_scorer.py", line 308, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^

,estimator,Pipeline(step..._state=42)))])
,param_grid,"{'model__estimator__C': [1.0, 2.0]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


1. total_df에서 rawText 컬럼의 데이터들을 이용하여 Kkma를 이용하여 문장별로 나눠준다. 
2. grid의 best_estimator_에서 예측을 실행 
3. 실행된 결과 값을 이용하여 데이터프레임( RawText, Aspect_pred, Pola_pred )으로 생성 
4. rawText, Aspect_pred 값을 이용하여 그룹화 -> 그룹화 연산에는 평균 

In [32]:
best_model = grid.best_estimator_

In [33]:
total_df.columns

Index(['Index', 'RawText', 'Source', 'Domain', 'MainCategory', 'ProductName',
       'Syllable', 'Word', 'GeneralPolarity', 'Aspects'],
      dtype='object')

In [34]:
total_df.loc[0, 'RawText']

'안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 딱 좋을 인생 경량 패딩 하나 소개해 드릴게요.  다 함께 go go go~~  제가 입어보고 좋아서 엄마께도 구매해드렸네요. 딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서 어디서 샀냐고 많이들 물어보는 옷입니다.  엄마도 제가 입은 것 보시더니 탐내셔서  주문해드렸어요. 영하로 내려가는 날씨에는 이것만 입기엔 얇지만 초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요. 색상도 디자인도 무난해서 사무실에 두고 입기도 좋고, 평소에 가볍게 나갈때 코디하기도 좋아요. 앞으로 코트를 입거나 할때 안에 내피로 활용하기도 딱일 듯요! 맘같아선 색깔별로 쟁이고 싶습니다.  20대와 50대 모두 아우르는 디자인이 무난하고 깔끔하면서 고급스러운 옷 추천합니다~~~  설명 끝! 좀 도움이 되셨나요? 그러면 좋아요 꾹 눌러 주시고 전 이만 총총총~~ 항상 여러 이웃님들께 감사드립니다. '

In [35]:
from konlpy.tag import Kkma

In [36]:
kkma = Kkma()
# raw_list에는 리뷰 문단을 문장으로 나눈 리스트를 담기 위한 공간
raw_list = []
# raw_dict 리뷰 문단마다 index를 키값으로 value는 리뷰 문단
raw_dict = {}
for i in range(len(total_df)):
    # print(kkma.sentences(total_df.loc[i, 'RawText']))
    # break
    raw_list.append(kkma.sentences(total_df.loc[i, 'RawText']))
    raw_dict[i] = total_df.loc[i, 'RawText']

In [37]:
sentence_df = pd.DataFrame()
for idx, raw in enumerate(raw_list):
    # print(raw)
    pred = best_model.predict(raw)
    # print(pred)
    temp_df = pd.DataFrame(pred, columns = [
        'Aspect_pred', 'Pola_pred'])
    temp_df['RawText'] = raw_dict[idx]
    # display(temp_df)
    sentence_df = pd.concat([sentence_df, temp_df])
    # break

In [38]:
len(sum(raw_list, []))

12883

In [39]:
sentence_df

,Aspect_pred,Pola_pred,RawText
0,1,1,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...
1,1,1,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...
2,1,1,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...
3,17,1,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...
4,5,0,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...
...,...,...,...
6,16,-1,홈쇼핑 보면서 고민고민하다가 구매한 옷인데요~ 어떤 옷인지 소개해드릴께요~ 봄 가...
7,5,1,홈쇼핑 보면서 고민고민하다가 구매한 옷인데요~ 어떤 옷인지 소개해드릴께요~ 봄 가...
8,5,1,홈쇼핑 보면서 고민고민하다가 구매한 옷인데요~ 어떤 옷인지 소개해드릴께요~ 봄 가...
9,6,1,홈쇼핑 보면서 고민고민하다가 구매한 옷인데요~ 어떤 옷인지 소개해드릴께요~ 봄 가...


In [40]:
group_df = sentence_df.groupby(['RawText', 
                                'Aspect_pred']).mean()

In [41]:
group_df.reset_index(inplace=True)

In [42]:
group_df['Aspect_pred'] = le.inverse_transform(group_df[
    'Aspect_pred'])

In [43]:
group_df.index

RangeIndex(start=0, stop=8362, step=1)

In [44]:
total_df.index

RangeIndex(start=0, stop=1423, step=1)

In [45]:
# total_df와 group_df를 조인 결합 
review_df = pd.merge(total_df, group_df, on = 'RawText', how = 'inner')

In [46]:
review_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8362 entries, 0 to 8361
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            8362 non-null   int64  
 1   RawText          8362 non-null   object 
 2   Source           8362 non-null   object 
 3   Domain           8362 non-null   object 
 4   MainCategory     8362 non-null   object 
 5   ProductName      8362 non-null   object 
 6   Syllable         8362 non-null   int64  
 7   Word             8362 non-null   int64  
 8   GeneralPolarity  8330 non-null   float64
 9   Aspects          8362 non-null   object 
 10  Aspect_pred      8362 non-null   object 
 11  Pola_pred        8362 non-null   float64
dtypes: float64(2), int64(3), object(7)
memory usage: 784.1+ KB


In [47]:
# ProductName의 빈도 수 체크 
len(review_df['ProductName'].unique())

517

In [49]:
# 제품별 리뷰의 상세 감정 분석이 가능
# 제품 이름 중 가장 많은 리뷰를 가진 제품을 선택하여 감정 점수의 평균을 확인
review_df.groupby(['ProductName', 'RawText'])

In [50]:
pd.pivot_table(
    data = review_df.drop_duplicates('RawText'),
    index = 'ProductName', 
    values = 'RawText', 
    aggfunc= ('count')
).sort_values('RawText',ascending=False)

,RawText
ProductName,
OO 코튼 가디건,14
OO 아** 구스코트,13
OO 브** 터틀넥 긴팔티,13
OO 폭스퍼 후드 구스 다운,12
OO 싱글 블레이저,11
...,...
OO 기모 부츠컷팬츠,1
OO 기모 본딩팬츠,1
OO 기모 본딩 팬츠,1


In [51]:
product_name = 'OO 코튼 가디건 '
# 해당 제품의 리뷰들의 전체적인 감정의 점수를 출력
# case -> ProductName 에서 필터링을 한 뒤 Aspect_pred 를 
# 기준으로 그룹화 -> Pola_pred의 평균 
test_df = review_df.loc[review_df['ProductName'] == product_name]
test_df.groupby('Aspect_pred')['Pola_pred'].mean()

Aspect_pred
가격      0.875000
기능      0.944444
길이      1.000000
두께      0.875000
디자인     1.000000
마감      1.000000
사이즈     0.300000
색상      0.950000
소재      1.000000
신축성     1.000000
제품구성   -1.000000
착용감     1.000000
촉감      1.000000
품질      0.600000
핏       1.000000
활용성     0.666667
Name: Pola_pred, dtype: float64

In [54]:
# case2 -> review_df에서 ProductNames과 Aspect_pred를 기준으로 그룹화
# Pola_pred의 평균을 구한다.
# 원하는 제품명을 선택하여 확인
group_df2 = review_df.groupby(['ProductName','Aspect_pred'])['Pola_pred'].mean()

In [55]:
group_df2[product_name]

Aspect_pred
가격      0.875000
기능      0.944444
길이      1.000000
두께      0.875000
디자인     1.000000
마감      1.000000
사이즈     0.300000
색상      0.950000
소재      1.000000
신축성     1.000000
제품구성   -1.000000
착용감     1.000000
촉감      1.000000
품질      0.600000
핏       1.000000
활용성     0.666667
Name: Pola_pred, dtype: float64